# Phys 235 Schwinger VQS Main Skeleton

This notebook is the project entry point. It owns concrete parameter choices, fixture I/O, workflow ordering, validation display, and report-ready plots. Module `.py` files only define algorithms and callable interfaces.

## 1. Dependencies and Local Imports

In Google Colab, run the install command if dependencies are missing.

In [ ]:
# Uncomment in Colab if needed.
# %pip install -q pennylane scipy matplotlib

from pathlib import Path
import importlib
import json
import sys

import numpy as onp
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / "module_4_vqe_quench.py").exists():
    CODE_DIR = cwd
elif (cwd / "code" / "module_4_vqe_quench.py").exists():
    CODE_DIR = cwd / "code"
else:
    CODE_DIR = Path("235_Final_Project") / "code"
PROJECT_ROOT = CODE_DIR.parent
TEST_DATA_ROOT = PROJECT_ROOT / "test_data"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import module_4_vqe_quench as module4_lib
module4_lib = importlib.reload(module4_lib)

from module_4_vqe_quench import (
    Module4Config,
    build_schwinger_hamiltonian,
    compute_observables,
    exact_time_evolution,
    hamiltonian_matrix,
    module4_acceptance_passed,
    run_module4_from_config,
)


## 2. Project Configuration

All concrete physics and optimizer choices live here. The source paper uses `q_final = 2` for the post-quench external field and `layer_count = 5` for the HVA depth.

In [ ]:
PHYSICS_CONFIG = {
    "N": 4,
    "ag": 1.0,
    "m_over_g": 1.0,
    "q_initial": 0.0,
    "q_final": 2.0,
    "g": 1.0,
    "layer_count": 5,
}

VQE_REPRO_CONFIG = {
    "n_restarts": 20,
    "seed": 1234,
    "learning_rate": 0.05,
    "max_steps": 3000,
    "grad_tol": 1e-4,
    "stall_window": 100,
    "stall_tol": 1e-9,
    "use_lbfgs_polish": False,
}

VQE_FIXTURE_CONFIG = {
    **VQE_REPRO_CONFIG,
    "n_restarts": 5,
    "max_steps": 120,
}

# For a final reproduction run, use VQE_REPRO_CONFIG. For fast fixture regeneration, use VQE_FIXTURE_CONFIG.
ACTIVE_VQE_CONFIG = VQE_REPRO_CONFIG
USE_TEST_DATA_INPUT = True
REGENERATE_TEST_DATA = False

MODULE4_FIXTURE_DIR = TEST_DATA_ROOT / "module_4"
MODULE4_METADATA_PATH = MODULE4_FIXTURE_DIR / "metadata.json"
MODULE4_ARRAYS_PATH = MODULE4_FIXTURE_DIR / "arrays.npz"


## 3. Test Data API

Later modules should use this API when they need upstream statevectors, Hamiltonians, optimized VQE parameters, or smoke-test inputs without rerunning expensive VQE.

In [ ]:
def make_module4_config(vqe_config):
    config = Module4Config(**PHYSICS_CONFIG, **vqe_config)
    config.validate()
    return config


def module4_summary(result):
    lines = [
        "Module 4 VQE + quench setup",
        f"N: {result.config.N}",
        f"ag: {result.config.ag}",
        f"m_over_g: {result.config.m_over_g}",
        f"q_initial: {result.config.q_initial}",
        f"q_final: {result.config.q_final}",
        f"layer_count: {result.config.layer_count}",
        f"best_energy: {result.vqe.best_energy:.12g}",
        f"exact_ground_energy: {result.vqe.exact_ground_energy:.12g}",
        f"r(E): {result.vqe.r_E:.12g}",
        f"ground_state_fidelity: {result.validation['ground_state_fidelity']:.12g}",
        f"post_quench_energy_q2: {result.quench.initial_energy_q2:.12g}",
        f"post_quench_variance_q2: {result.quench.initial_variance_q2:.12g}",
        f"acceptance_passed: {module4_acceptance_passed(result.validation)}",
    ]
    return "\n".join(lines)


def save_module4_test_data(result, metadata_path=MODULE4_METADATA_PATH, arrays_path=MODULE4_ARRAYS_PATH):
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        "schema_version": 1,
        "module": "module_4",
        "description": "VQE q=0 ground-state and q=2 quench-ready fixture for smoke tests.",
        "source_paper": "Nagano, Bapat, Bauer, arXiv:2302.10933",
        "config": result.config.to_dict(),
        "validation": result.validation,
        "vqe": {
            "best_energy": result.vqe.best_energy,
            "adam_energy": result.vqe.adam_energy,
            "exact_ground_energy": result.vqe.exact_ground_energy,
            "exact_max_energy": result.vqe.exact_max_energy,
            "r_E": result.vqe.r_E,
            "polished": result.vqe.polished,
        },
        "quench": {
            "initial_energy_q0": result.quench.initial_energy_q0,
            "initial_energy_q2": result.quench.initial_energy_q2,
            "initial_variance_q2": result.quench.initial_variance_q2,
            "initial_observables_q2": result.quench.initial_observables_q2,
        },
        "arrays_file": arrays_path.name,
        "array_keys": [
            "theta_opt",
            "psi_0",
            "statevector",
            "exact_ground_state",
            "H_initial_matrix",
            "H_final_matrix",
        ],
    }
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    onp.savez_compressed(
        arrays_path,
        theta_opt=result.vqe.theta_opt,
        psi_0=result.quench.psi_0,
        statevector=result.vqe.statevector,
        exact_ground_state=result.vqe.exact_ground_state,
        H_initial_matrix=result.quench.H_initial_matrix,
        H_final_matrix=result.quench.H_final_matrix,
    )
    return metadata


def load_module_test_data(module_name, test_data_root=TEST_DATA_ROOT):
    module_dir = test_data_root / module_name
    metadata_path = module_dir / "metadata.json"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing test-data metadata: {metadata_path}")
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    if metadata.get("module") != module_name:
        raise ValueError(f"Metadata module mismatch: {metadata.get('module')} != {module_name}")
    arrays_path = module_dir / metadata["arrays_file"]
    if not arrays_path.exists():
        raise FileNotFoundError(f"Missing test-data arrays: {arrays_path}")
    arrays = onp.load(arrays_path, allow_pickle=False)
    return metadata, arrays


def validate_module4_fixture(metadata, arrays):
    required_keys = {"theta_opt", "psi_0", "H_initial_matrix", "H_final_matrix"}
    missing = required_keys.difference(arrays.files)
    if missing:
        raise ValueError(f"Module 4 fixture missing arrays: {sorted(missing)}")
    config = metadata["config"]
    dim = 2 ** int(config["N"])
    if tuple(arrays["psi_0"].shape) != (dim,):
        raise ValueError("psi_0 shape does not match N")
    if tuple(arrays["H_final_matrix"].shape) != (dim, dim):
        raise ValueError("H_final_matrix shape does not match N")
    if float(metadata["validation"]["r_E"]) <= 0.99:
        raise ValueError("Module 4 fixture does not satisfy r(E) > 0.99")
    return True


def load_module4_fixture():
    metadata, arrays = load_module_test_data("module_4")
    validate_module4_fixture(metadata, arrays)
    return metadata, arrays


## 4. Module 4: VQE Ground State and Quench Setup

Use fixture input for smoke tests, or regenerate the data live by setting `USE_TEST_DATA_INPUT = False` or `REGENERATE_TEST_DATA = True`.

In [ ]:
if USE_TEST_DATA_INPUT and MODULE4_METADATA_PATH.exists() and not REGENERATE_TEST_DATA:
    module4_metadata, module4_arrays = load_module4_fixture()
    module4_config = Module4Config(**module4_metadata["config"])
    print("Loaded Module 4 fixture from", MODULE4_METADATA_PATH)
    print("r(E):", module4_metadata["validation"]["r_E"])
else:
    module4_config = make_module4_config(ACTIVE_VQE_CONFIG)
    module4_result = run_module4_from_config(module4_config)
    print(module4_summary(module4_result))
    module4_metadata = save_module4_test_data(module4_result)
    module4_arrays = onp.load(MODULE4_ARRAYS_PATH, allow_pickle=False)
    print("Saved Module 4 fixture to", MODULE4_FIXTURE_DIR)


## 5. Validation Gate

Later modules should not run unless the live result or loaded fixture satisfies the Module 4 acceptance checks.

In [ ]:
if "module4_result" in globals():
    module4_ready = module4_acceptance_passed(module4_result.validation)
    validation_source = module4_result.validation
else:
    module4_ready = bool(float(module4_metadata["validation"]["r_E"]) > 0.99)
    validation_source = module4_metadata["validation"]

print("Module 4 ready for dynamics:", module4_ready)
for key, value in validation_source.items():
    print(f"{key}: {value}")

assert module4_ready, "Module 4 validation failed; do not proceed to Trotter/VQS yet."


## 6. Downstream Input API

This payload is the stable preinput for later modules: optimized parameters, the quench-ready state, and exact Hamiltonian matrices.

In [ ]:
module4_downstream_input = {
    "config": module4_metadata["config"],
    "theta_opt": module4_arrays["theta_opt"],
    "psi_0": module4_arrays["psi_0"],
    "H_initial_matrix": module4_arrays["H_initial_matrix"],
    "H_final_matrix": module4_arrays["H_final_matrix"],
    "validation": module4_metadata["validation"],
}

for key, value in module4_downstream_input.items():
    if hasattr(value, "shape"):
        print(key, value.shape)
    else:
        print(key, type(value))


## 7. Exact Reference Smoke Output

This is a small exact-evolution reference generated from the Module 4 downstream input. It is for interface validation; final plots should be produced after Trotter and VQS modules exist.

In [ ]:
times = onp.linspace(0.0, 1.0, 11)
states = exact_time_evolution(
    module4_downstream_input["H_final_matrix"],
    module4_downstream_input["psi_0"],
    times,
)

reference = {"times": times, "electric_field": [], "chiral_condensate": [], "charge": []}
for state in states:
    obs = compute_observables(
        state,
        N=module4_config.N,
        ag=module4_config.ag,
        external_field=module4_config.q_final,
        g=module4_config.g,
    )
    reference["electric_field"].append(obs["electric_field"])
    reference["chiral_condensate"].append(obs["chiral_condensate"])
    reference["charge"].append(obs["charge"])

for key in ["electric_field", "chiral_condensate", "charge"]:
    reference[key] = onp.asarray(reference[key], dtype=float)
    print(key, reference[key].shape)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(reference["times"], reference["electric_field"], marker="o", label="ED electric field")
plt.plot(reference["times"], reference["chiral_condensate"], marker="s", label="ED chiral condensate")
plt.xlabel("time")
plt.ylabel("observable")
plt.title(f"Module 4 exact reference, q={module4_config.q_final}, L={module4_config.layer_count}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
